# Code Changes with Executable Evidence

John Martinez

In [1]:
cases = [(0, 4, 0), (4, 4, 1), (5, 4, 2), (10, 4, 3)]

original = "\n".join([
    "def boxes_needed(items, capacity):",
    "    if type(items) is not int or type(capacity) is not int:",
    "        raise ValueError('whole integer quantities required')",
    "    if items < 0 or capacity <= 0:",
    "        raise ValueError('invalid quantity bounds')",
    "    return items // capacity",
]) + "\n"

print("Examples:", cases)
print(original)

Examples: [(0, 4, 0), (4, 4, 1), (5, 4, 2), (10, 4, 3)]
def boxes_needed(items, capacity):
    if type(items) is not int or type(capacity) is not int:
        raise ValueError('whole integer quantities required')
    if items < 0 or capacity <= 0:
        raise ValueError('invalid quantity bounds')
    return items // capacity



## Prediction

Ten items need three boxes. The original function uses floor division and will incorrectly return two boxes.

In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory

temporary = TemporaryDirectory(prefix="ag10-")

root = Path(temporary.name).resolve()

def project_path(name):

    if name not in {
        "shipping.py",
        "test_shipping.py"
    }:
        raise ValueError(
            "file not permitted"
        )

    target = (
        root / name
    ).resolve()

    if not target.is_relative_to(root):
        raise ValueError(
            "path outside project"
        )

    return target


def write_file(name, text):
    return project_path(name).write_text(
        text,
        encoding="utf-8"
    )


write_file(
    "shipping.py",
    original
)

print(
    "Files:",
    sorted(
        p.name
        for p in root.iterdir()
    )
)

Files: ['shipping.py']


## Prediction

Only shipping.py should exist. Filenames outside the allowlist should be rejected.

In [4]:
tests = """import unittest
from shipping import boxes_needed

class BoxTests(unittest.TestCase):

    def test_examples(self):

        for items, capacity, expected in [
            (0,4,0),
            (4,4,1),
            (5,4,2),
            (10,4,3)
        ]:

            with self.subTest(
                items=items,
                capacity=capacity
            ):

                self.assertEqual(
                    boxes_needed(
                        items,
                        capacity
                    ),
                    expected
                )

    def test_invalid(self):

        for items, capacity in [
            (-1,4),
            (2,0),
            (True,4),
            (2,True),
            (1.5,4)
        ]:

            with self.subTest(
                items=items,
                capacity=capacity
            ):

                with self.assertRaises(
                    ValueError
                ):

                    boxes_needed(
                        items,
                        capacity
                    )
"""

write_file(
    "test_shipping.py",
    tests
)

print(
    "Test methods:",
    tests.count(
        "    def test_"
    )
)

print(
    "Example cases:",
    len(cases)
)

Test methods: 2
Example cases: 4


## Prediction

The file contains two test methods. There are four valid example cases and five invalid-input cases.

In [5]:
import subprocess
import sys

def run_tests():

    completed = subprocess.run(
        [
            sys.executable,
            "-B",
            "-m",
            "unittest",
            "discover",
            "-v"
        ],
        cwd=root,
        capture_output=True,
        text=True,
        timeout=10,
    )

    summary = {
        "returncode":
            completed.returncode,
        "failed_subtests":
            completed.stderr.count(
                "FAIL:"
            ),
        "errors":
            completed.stderr.count(
                "ERROR:"
            )
    }

    return (
        summary,
        completed.stderr
    )


baseline, baseline_details = (
    run_tests()
)

print(
    "Original result:",
    baseline
)


Original result: {'returncode': 1, 'failed_subtests': 0, 'errors': 0}


## Prediction

The original function should fail two valid example cases because floor division undercounts partial boxes.

In [6]:
print(baseline_details)

test_examples (test_shipping.BoxTests.test_examples) ... 
  test_examples (test_shipping.BoxTests.test_examples) (items=5, capacity=4) ... FAIL
  test_examples (test_shipping.BoxTests.test_examples) (items=10, capacity=4) ... FAIL
test_invalid (test_shipping.BoxTests.test_invalid) ... ok

FAIL: test_examples (test_shipping.BoxTests.test_examples) (items=5, capacity=4)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\leoma\AppData\Local\Temp\ag10-kxkdviom\test_shipping.py", line 20, in test_examples
    self.assertEqual(
    ~~~~~~~~~~~~~~~~^
        boxes_needed(
        ^^^^^^^^^^^^^
    ...<3 lines>...
        expected
        ^^^^^^^^
    )
    ^
AssertionError: 1 != 2

FAIL: test_examples (test_shipping.BoxTests.test_examples) (items=10, capacity=4)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\leoma\AppData\Local\Temp\ag10-kxkdviom\

## Prediction

The test output should show failures for the partial box examples. Five items and ten items with capacity four should not match the expected results.

In [8]:
repaired, repaired_details = (
    run_tests()
)

print(
    "Repaired result:",
    repaired
)

Repaired result: {'returncode': 0, 'failed_subtests': 0, 'errors': 0}


In [7]:
fixed_source = "\n".join([
    "def boxes_needed(items, capacity):",
    "    if type(items) is not int or type(capacity) is not int:",
    "        raise ValueError('whole integer quantities required')",
    "    if items < 0 or capacity <= 0:",
    "        raise ValueError('invalid quantity bounds')",
    "    return (items + capacity - 1) // capacity",
]) + "\n"

write_file(
    "shipping.py",
    fixed_source
)

print(fixed_source)

def boxes_needed(items, capacity):
    if type(items) is not int or type(capacity) is not int:
        raise ValueError('whole integer quantities required')
    if items < 0 or capacity <= 0:
        raise ValueError('invalid quantity bounds')
    return (items + capacity - 1) // capacity



## Prediction

The revised formula uses ceiling-style integer arithmetic and should correctly handle partial boxes, zero items, and exact multiples.

In [9]:
print(repaired_details)

test_examples (test_shipping.BoxTests.test_examples) ... ok
test_invalid (test_shipping.BoxTests.test_invalid) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK



## Prediction

The repaired test run should show all tests passing with no failures or errors.

In [10]:
handoff = {
    "change":
        "Replaced floor division with ceiling-style integer arithmetic.",
    "verification":
        "All authored tests passed.",
    "scope":
        "Validated against provided examples and invalid inputs."
}

for key, value in handoff.items():
    print(
        key + ":",
        value
    )

change: Replaced floor division with ceiling-style integer arithmetic.
verification: All authored tests passed.
scope: Validated against provided examples and invalid inputs.


# Reflection

This activity demonstrated how coding agents can use testing and execution feedback to evaluate changes. The original implementation failed because floor division undercounted partial boxes. After changing the arithmetic formula and rerunning the tests, all authored tests passed. The handoff summarized the modification, supporting evidence, and scope of validation.